In [1]:
import numpy as np
import cupy as cp
import scipy
from typing import Callable, Any
import quadpy
import time

num_points = 10**7
points = np.random.randn(num_points, 3)
magnitudes = np.linalg.norm(points, axis=1, keepdims=True)
normalized_points = points / magnitudes
hull = scipy.spatial.ConvexHull(normalized_points)

print(f"Hull has {len(hull.simplices)} simplices.")

Hull has 19999996 simplices.


In [2]:
def hull_surface_integral(
    function: Callable,
    hull: scipy.spatial.ConvexHull,
    scheme: Any | None = None,
    use_gpu: bool = False,
) -> np.ndarray | cp.ndarray:
    """Compute the integral of a function over the surface of a convex hull. It
    is assumed that the function returns a scalar output, i.e. maps R^n -> R.

    function: The function to be integrated. This function must be vectorized
    to accept arguments of shape

    N x n

    where n-1 is the number of dimensions of the simplical facets of the
    surface (alternatively, n is the number of dimensions of the ambient space
    in which the surface lies. For example, for the surface of a sphere, n=3)

    hull: The convex hull object that defines the surface.
    scheme: Integration scheme to be used.
    use_gpu: If true, use cupy instead of numpy
    """
    # Pick appropriate array module
    xp = cp if use_gpu else np

    # Pick integration scheme
    # For future developers: one can use Cayley-Menger determinants for d > 2
    num_dimensions = hull.points.shape[1] - 1
    if num_dimensions > 2:
        raise NotImplementedError(
            "Integration of surfaces in d > 2 dimensions"
            "not currently supported."
        )

    if scheme is None:
        # The second parameter to quadpy's scheme method here is about
        # accuracy of the scheme, not the number of spatial dimensions.
        scheme = quadpy.tn.grundmann_moeller(num_dimensions, 3)

    barycentric_weights = xp.asarray(scheme.points)
    weights = xp.asarray(scheme.weights)

    # Generate integration points
    num_simplices = len(hull.simplices)
    num_weights = len(weights)
    simplical_points = xp.asarray(hull.points)[
        xp.asarray(hull.simplices)
    ].transpose(0, 2, 1)
    integration_points = simplical_points @ barycentric_weights
    reshaped_points = integration_points.transpose(0, 2, 1).reshape(
        -1, num_dimensions + 1
    )

    # Find simplex volumes using cross product
    v1s = simplical_points[:, :, 1] - simplical_points[:, :, 0]
    v2s = simplical_points[:, :, 2] - simplical_points[:, :, 0]
    cross_products = xp.cross(v1s, v2s)
    areas = 0.5 * xp.sqrt(xp.sum(cross_products**2, axis=1))

    # Output
    function_output = function(reshaped_points)
    weighted_output = (
        function_output
        * xp.tile(weights, num_simplices)
        * xp.repeat(areas, num_weights)
    )
    integral = xp.sum(weighted_output)
    return integral

In [3]:
def hull_surface_integral_vector(
    function: Callable,
    hull: scipy.spatial.ConvexHull,
    scheme: Any | None = None,
    use_gpu: bool = False,
) -> np.ndarray | cp.ndarray:
    """Compute the integral of a function over the surface of a convex hull. It
    is assumed that the function returns a vector output, i.e. maps R^3 -> R^3.

    function: The function to be integrated. This function must be vectorized
    to accept arguments of shape

    N x n

    where n-1 is the number of dimensions of the simplical facets of the
    surface (alternatively, n is the number of dimensions of the ambient space
    in which the surface lies. For example, for the surface of a sphere, n=3)

    hull: The convex hull object that defines the surface.
    scheme: Integration scheme to be used.
    use_gpu: If true, use cupy instead of numpy
    """
    # Pick appropriate array module
    xp = cp if use_gpu else np

    # Pick integration scheme
    # For future developers: one can use Cayley-Menger determinants for d > 2
    num_dimensions = hull.points.shape[1] - 1
    if num_dimensions > 2:
        raise NotImplementedError(
            "Integration of surfaces in d > 2 dimensions"
            "not currently supported."
        )

    if scheme is None:
        # The second parameter to quadpy's scheme method here is about
        # accuracy of the scheme, not the number of spatial dimensions.
        scheme = quadpy.tn.grundmann_moeller(num_dimensions, 3)

    barycentric_weights = xp.asarray(scheme.points)
    weights = xp.asarray(scheme.weights)
    points = xp.asarray(hull.points)
    vertices = xp.asarray(hull.vertices)
    centroid = xp.mean(points[vertices], axis=0)
    simplices = xp.asarray(hull.simplices)

    # Generate integration points
    num_simplices = len(hull.simplices)
    num_weights = len(weights)
    simplical_points = points[simplices].transpose(0, 2, 1)
    integration_points = simplical_points @ barycentric_weights
    reshaped_points = integration_points.transpose(0, 2, 1).reshape(
        -1, num_dimensions + 1
    )

    # Find simplex volumes using cross product
    v1s = simplical_points[:, :, 1] - simplical_points[:, :, 0]
    v2s = simplical_points[:, :, 2] - simplical_points[:, :, 0]
    cross_products = xp.cross(v1s, v2s)
    unit_normals = cross_products / xp.linalg.norm(
        cross_products, axis=1, keepdims=True
    )
    radial_points = simplical_points[:, :, 0] - centroid
    dot_products = xp.sum(radial_points * unit_normals, axis=1)
    orientation_signs = xp.sign(dot_products)
    areas = 0.5 * xp.sqrt(xp.sum(cross_products**2, axis=1))

    # Output
    function_output = function(reshaped_points)
    integrand = xp.sum(
        function_output.reshape(num_simplices, num_weights, 3)
        * unit_normals[:, None, :],
        axis=2,
    ).reshape(num_simplices * num_weights)
    weighted_output = (
        integrand
        * xp.tile(weights, num_simplices)
        * xp.repeat(areas, num_weights)
        * xp.repeat(orientation_signs, num_weights)
    )
    integral = xp.sum(weighted_output)
    return integral

In [4]:
def identity_vector(x):
    return (
        np.repeat(x[:, 0] ** 2, 3).reshape(len(x), 3)
        * x
        / np.linalg.norm(x, axis=1, keepdims=True)
    )


def identity_vector_cp(x):
    return (
        cp.repeat(x[:, 0] ** 2, 3).reshape(len(x), 3)
        * x
        / cp.linalg.norm(x, axis=1, keepdims=True)
    )


s = time.perf_counter()
integral = hull_surface_integral_vector(identity_vector, hull, use_gpu=False)
e = time.perf_counter()
print(f"Time elapsed CPU: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area/3}")
print(f"Error one: {hull.area/3 - integral}")
print("---")
s = time.perf_counter()
integral = hull_surface_integral_vector(identity_vector_cp, hull, use_gpu=True)
e = time.perf_counter()
print(f"Time elapsed GPU: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area/3}")
print(f"Error one: {hull.area/3 - integral}")
print("---")
s = time.perf_counter()
integral = hull_surface_integral_vector(identity_vector_cp, hull, use_gpu=True)
e = time.perf_counter()
print(f"Time elapsed: GPU second: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area/3}")
print(f"Error one: {hull.area/3 - integral}")
print("---")

Time elapsed CPU: 71.6754966811277
Calculated area: 4.188783500924168
Actual area: 4.188787691100991
Error one: 4.190176823293257e-06
---
Time elapsed GPU: 2.460176580119878
Calculated area: 4.188783500924164
Actual area: 4.188787691100991
Error one: 4.1901768268459705e-06
---
Time elapsed: GPU second: 0.1463369382545352
Calculated area: 4.188783500924164
Actual area: 4.188787691100991
Error one: 4.1901768268459705e-06
---


In [5]:
# Surface area of the sphere
# def identity_function_one(x):
#     return np.ones(len(x))


# def identity_function_two(x):
#     return x[:, 0] ** 2 + x[:, 1] ** 2 + x[:, 2] ** 2


# integral_one = hull_surface_integral(identity_function_one, hull)
# integral_two = hull_surface_integral(identity_function_two, hull)

# print(f"Calculated area one: {integral_one}")
# print(f"Calculated area two: {integral_two}")
# print(f"Actual area: {hull.area}")
# print(f"Error one: {hull.area - integral_one}")
# print(f"Error two: {hull.area - integral_two}")

In [6]:
# Surface area of the sphere with cupy
# def identity_function_one(x):
#     return cp.ones(len(x))


# def identity_function_two(x):
#     return x[:, 0] ** 2 + x[:, 1] ** 2 + x[:, 2] ** 2


# integral_one = hull_surface_integral(identity_function_one, hull, use_gpu=True)
# integral_two = hull_surface_integral(identity_function_two, hull, use_gpu=True)

# print(f"Calculated area one: {integral_one}")
# print(f"Calculated area two: {integral_two}")
# print(f"Actual area: {hull.area}")
# print(f"Error one: {hull.area - integral_one}")
# print(f"Error two: {hull.area - integral_two}")

In [7]:
# Surface area of the sphere with cupy
# def function(x):
#     return x[:, 0] ** 2


# integral = hull_surface_integral(function, hull, use_gpu=True)

# print(f"Calculated: {integral}")
# print(f"Actual: {4*np.pi/3}")
# print(f"Error two: {4*np.pi/3 - integral}")